In [ ]:
# --- Xray Cloud: Release Test Metrics with Smart Parquet Caching (FINAL FIX) ---

import requests, getpass, os, datetime, pandas as pd, urllib3
from datetime import datetime as dt

# ============================================================
# Global Config
# ============================================================

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
CACHE_DIR = "./xray_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

XRAY_AUTH_URL    = "https://xray.cloud.getxray.app/api/v2/authenticate"
XRAY_GRAPHQL_URL = "https://xray.cloud.getxray.app/api/v2/graphql"
VERIFY_SSL = False

# ============================================================
# Auth
# ============================================================

CID  = getpass.getpass("XRAY_CLIENT_ID: ")
CSEC = getpass.getpass("XRAY_CLIENT_SECRET: ")

def auth():
    """Authenticate and return token."""
    payload = {"client_id": CID, "client_secret": CSEC}
    print("Authenticating to Xray Cloud...")
    r = requests.post(XRAY_AUTH_URL, json=payload, timeout=30, verify=VERIFY_SSL)
    if r.status_code != 200:
        print("Authentication failed:", r.text)
        raise SystemExit("Check Client ID/Secret or environment.")
    return r.text.strip('"')

TOKEN   = auth()
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
# ============================================================
# Helper: GraphQL Wrapper
# ============================================================

def gql(query, variables=None):
    """Execute GraphQL query."""
    # NOTE: Added a check for the dummy token to prevent actual API calls
    if TOKEN == "DUMMY_TOKEN_FOR_STRUCTURE_VALIDATION":
        print(" Using dummy token. Skipping actual GraphQL request.")
        
        # FIXED: Ensure dictionary returns are syntactically complete
        if "getTestPlans" in query:
             return {"getTestPlans": {"results": [{"issueId": "12345", "jira": {"key": "PROJ-1003", "summary": "Release A", "status": "Done"}}]}}
        elif "getTestPlan" in query and "tests" in query:
             return {"getTestPlan": {"tests": {"total": 3, "start": 0, "limit": 100, "results": [
                 {"jira": {"key": "TEST-1"}, "status": {"name": "PASSED"}},
                 {"jira": {"key": "TEST-2"}, "status": {"name": "FAILED"}},
                 {"jira": {"key": "TEST-3"}, "status": {"name": "TO DO"}}
             ]}}}
        return {}


    r = requests.post(
        XRAY_GRAPHQL_URL,
        json={"query": query, "variables": variables or {}},
        headers=HEADERS,
        timeout=60,
        verify=VERIFY_SSL,
    )
    
    # CRITICAL: Log response content for 400 Bad Request
    if r.status_code == 400:
        print(f"\n!!! HTTP 400 BAD REQUEST ERROR !!!")
        print(f"XRAY RESPONSE TEXT: {r.text}")
        print(f"QUERY ATTEMPTED: {query.splitlines()[1].strip()}...")
        print(f"VARIABLES USED: {variables}")
        r.raise_for_status() 

    r.raise_for_status()
    data = r.json()
    if "errors" in data:
        raise SystemExit(f"GraphQL errors: {data['errors']}")
    return data["data"]

# ============================================================
# Status Normalization (EXPANDED FOR ACCURACY)
# ============================================================

# EXPANDED STATUS MAP to handle common Xray/Jira statuses
STATUS_MAP = {
    # PASSED
    "PASSED": "PASSED", "PASS": "PASSED", "OK": "PASSED",
    
    # FAILED
    "FAILED": "FAILED", "FAIL": "FAILED",
    
    # TO DO (Not executed yet)
    "TO DO": "TO DO", "TODO": "TO DO", "NOT_RUN": "TO DO", "UNTESTED": "TO DO",
    "NOT EXECUTED": "TO DO", "NOT RUN": "TO DO",
    
    # BLOCKED
    "BLOCKED": "BLOCKED",
    
    # EXECUTING (In progress)
    "EXECUTING": "EXECUTING", "WIP": "EXECUTING", "IN PROGRESS": "EXECUTING",
    
    # DESCOPED (Removed from scope/wont do)
    "DESCOPED": "DESCOPED", "SKIPPED": "DESCOPED", "WONT DO": "DESCOPED", 
    "ABORTED": "DESCOPED", "CANCELLED": "DESCOPED"
}

def norm_status(s: str) -> str:
    """Normalize status name to a standard set."""
    # Ensure the status is mapped. If it's a new status, it returns the original name 
    # but still upper-cased and stripped for consistency, allowing it to be seen.
    return STATUS_MAP.get((s or "").strip().upper(), (s or "").strip().upper())

# ============================================================
# Helper: Save to Cache
# ============================================================

def cache_path_for(plan_key): 
    return os.path.join(CACHE_DIR, f"{plan_key}.parquet")

def save_to_cache(plan_key, df):
    path = cache_path_for(plan_key)
    df.to_parquet(path, index=False)
    print(f" Saved data for {plan_key} → {path}")

# ============================================================
# Helper: Get Test Plan Status
# ============================================================

def get_test_plan_status(plan_key):
    """Return the Jira/Xray status of the given Test Plan."""
    Q_STATUS = """
    query($jql:String!, $limit:Int!, $start:Int!) {
      getTestPlans(jql:$jql, limit:$limit, start:$start) {
        results {
          jira(fields:["key","summary","status"])
        }
      }
    }
    """
    d = gql(Q_STATUS, {"jql": f'key = "{plan_key}"', "limit": 1, "start": 0})
    plans = d.get("getTestPlans", {}).get("results", [])
    if not plans:
        return "Unknown"

    # Access the first item in the list [0]
    status_field = plans[0]["jira"].get("status")

    if isinstance(status_field, dict):
        return status_field.get("name", "Unknown").strip().title()
    elif isinstance(status_field, str):
        return status_field.strip().title()
    else:
        return "Unknown"

# ============================================================
# Fetch Release Data (Accurate Status from Test Plan's Tests)
# ============================================================

# CORRECTED QUERY: Uses jira(fields:["key"]) directly under tests.results
Q_PLAN_TEST_STATUS = """
query($planId:String!, $limit:Int!, $start:Int!) {
  getTestPlan(issueId:$planId) {
    jira(fields:["key", "summary", "status"])
    tests(limit:$limit, start:$start) {
      total start limit
      results {
        status { name }
        jira(fields:["key"])
      }
    }
  }
}
"""

def fetch_release_data(plan_key):
    print(f" Fetching data for {plan_key} ...")

    # --- 1) Resolve Plan Info & Fetch All Tests with Status (Paginated) ---
    all_tests_data = []
    start, limit = 0, 100
    plan_id = None
    plan_name = None
    plan_status = None
    total_tests = 0

    # First, get the issueId for the Plan
    Q_PLAN_ID = """
    query($jql:String!, $limit:Int!, $start:Int!) {
      getTestPlans(jql:$jql, limit:$limit, start:$start) {
        results { issueId }
      }
    }
    """
    plans = gql(Q_PLAN_ID, {"jql": f'key = "{plan_key}"', "limit": 1, "start": 0})
    if not plans["getTestPlans"]["results"]:
        print(f"Error: Test Plan {plan_key} not found.")
        return pd.DataFrame()

    plan_id = plans["getTestPlans"]["results"][0]["issueId"]

    # Now, paginate to get all Tests and their statuses
    while True:
        d = gql(Q_PLAN_TEST_STATUS, {"planId": plan_id, "limit": limit, "start": start})
        
        # Extract Plan Info from the first page of the response
        if start == 0:
            plan_jira = d["getTestPlan"]["jira"]
            plan_name = plan_jira["summary"]
            plan_status = plan_jira["status"]

        tests_data = d["getTestPlan"]["tests"]
        total_tests = tests_data["total"]

        for test_result in tests_data["results"]:
            # Access "jira" directly
            test_key = (test_result.get("jira", {})).get("key") 
            sname = (test_result.get("status") or {}).get("name") or "UNKNOWN"
            
            if test_key:
                all_tests_data.append({
                    "plan_key": plan_key,
                    "plan_name": plan_name,
                    "plan_status": plan_status,
                    "test_key": test_key,
                    "status": norm_status(sname)
                })
        
        start += tests_data["limit"]
        if start >= total_tests:
            break

    combined = pd.DataFrame(all_tests_data)
    combined["timestamp"] = datetime.datetime.now().isoformat()
    # Data is now one row per planned test, with the official current status
    return combined

# ============================================================
# Dyanmic Cache Logic
# ============================================================

def fetch_or_load(plan_key):
    """Smart cache loader that refreshes if plan is not Done/Closed."""
    path = cache_path_for(plan_key)
    plan_status = get_test_plan_status(plan_key)
    print(f" Plan {plan_key} status: {plan_status}")

    # Normalize status (case-insensitive)
    plan_status_norm = (plan_status or "").strip().lower()

    # Define what counts as "Done/finalized"
    done_statuses = {"done", "closed", "completed"}
    
    # If not Done → always fetch fresh data
    if plan_status_norm not in done_statuses:
        print(f" Plan is active ({plan_status}) → fetching fresh data from Xray...")
        df = fetch_release_data(plan_key)
        save_to_cache(plan_key, df)
        return df

    # If Done/finalized & cache exists → reuse cached data
    if os.path.exists(path):
        try:
            cached = pd.read_parquet(path)
            print(f" Using cached data for {plan_key} (status = {plan_status})")
            return cached
        except Exception as e:
            print(f" Cache load failed for {plan_key}: {e} → refetching...")
            df = fetch_release_data(plan_key)
            save_to_cache(plan_key, df)
            return df

    # If Done/finalized but no cache yet → fetch once and cache
    print(f" No cache found for {plan_key} (status = {plan_status}) → fetching fresh data...")
    df = fetch_release_data(plan_key)
    save_to_cache(plan_key, df)
    return df


# ============================================================
# Compute Release Metrics (All 6 Statuses) - SIMPLIFIED LOGIC
# ============================================================

def compute_release_metrics(df_all):
    """Compute release metrics using the accurate status data."""
    summary = []
    
    # Define a default set of statuses to ensure all columns exist
    status_list = ["TO DO", "PASSED", "FAILED", "BLOCKED", "EXECUTING", "DESCOPED"]

    for rel in df_all["plan_key"].unique():
        subset = df_all[df_all["plan_key"] == rel].copy()

        # --- Resolve Plan Status ---
        plan_status = subset["plan_status"].iloc[0]
        if isinstance(plan_status, dict):
             plan_status = plan_status.get("name", "Unknown").strip().title()
        elif isinstance(plan_status, str):
             plan_status = plan_status.strip().title()
        else:
             plan_status = "Unknown"

        # The DataFrame now contains exactly one row per unique planned test,
        # and the status column is the official Xray status for that test in the plan.
        
        # --- Planned scope (Total unique tests in the plan) ---
        planned = subset["test_key"].nunique()

        # --- Unique outcomes (Counts by Status) ---
        # The value_counts result is now the authoritative set of metrics
        status_counts = subset["status"].value_counts().to_dict()
        
        # Filter status_counts to include ONLY the 6 standard statuses
        counts = {s: status_counts.get(s, 0) for s in status_list}
        
        # Log any unrecognized statuses for troubleshooting
        unrecognized_statuses = sum([v for k, v in status_counts.items() if k not in status_list])
        if unrecognized_statuses > 0:
            print(f"Warning: {unrecognized_statuses} tests in {rel} mapped to unrecognized statuses.")

        todo, passed, failed, blocked, executing, descoped = (
            counts["TO DO"], counts["PASSED"], counts["FAILED"],
            counts["BLOCKED"], counts["EXECUTING"], counts["DESCOPED"]
        )
        
        # Executed tests are Planned minus "TO DO"
        executed_tests = planned - (todo + descoped + blocked)

        # --- Metrics Calculation ---
        total_executed_for_rate = passed + failed
        pass_rate = round((passed / total_executed_for_rate * 100), 2) if total_executed_for_rate else 0
        
        # Completion is based on all non-TO DO statuses
        completion_count = passed + failed + descoped + blocked + executing
        
        exec_coverage = round((executed_tests / planned * 100), 2) if planned else 0
        completion = round((completion_count / planned * 100), 2) if planned else 0
        remaining = round((todo / planned * 100), 2) if planned else 0

        summary.append({
            "Release": rel,
            "Status": plan_status,
            "Planned Tests": planned,
            # NEW: Add the count of tests that are not "TO DO"
            "Executed Tests Count": executed_tests,
            "Execution Coverage %": exec_coverage,
            "Pass %": pass_rate,
            "Completion %": completion,
            "Remaining %": remaining,
            "To Do": todo,
            "Passed": passed,
            "Failed": failed,
            "Blocked": blocked,
            "Executing": executing,
            "Descoped": descoped
        })

    cols = [
        "Release", "Status", "Planned Tests", "Executed Tests Count", # New column added here
        "Execution Coverage %", "Pass %", "Completion %", "Remaining %",
        "To Do", "Passed", "Failed", "Blocked", "Executing", "Descoped"
    ]
    return pd.DataFrame(summary)[cols]


# ============================================================
# Run for Releases
# ============================================================

release_keys = [
    "PROJ-1003", "PROJ-1001", "PROJ-1002"  # add more release plan keys here 
]

all_dfs = [fetch_or_load(rk) for rk in release_keys]
df_all = pd.concat(all_dfs, ignore_index=True)

dashboard_df = compute_release_metrics(df_all)
print("\n=== Release Test Metrics Dashboard ===\n")
print(dashboard_df.to_string(index=False))

dashboard_df.to_csv("xray_cache/release_metrics_dashboard.csv", index=False)
dashboard_df.to_parquet("xray_cache/release_metrics_dashboard.parquet", index=False)
print("\n Dashboard successfully generated.")
